In [1]:
import pandas as pd
df = pd.read_csv("Telco_Customer_Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df = df.drop(columns=["customerID"])
df["Churn"] = (df["Churn"] == "Yes").astype(int)
print(df.shape)

(7043, 20)


In [2]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=["Churn"])
y = df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print("Train:", X_train.shape, "| Test:", X_test.shape)
print("ti le churn - train:", round(y_train.mean(), 4), "| test:", round(y_test.mean(), 4))

Train: (5634, 19) | Test: (1409, 19)
ti le churn - train: 0.2654 | test: 0.2654


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
binary_cols = ["SeniorCitizen"]
cat_cols = [c for c in X.columns if c not in num_cols + binary_cols]
print("So cot chu:", len(cat_cols))

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ],
    remainder = "passthrough",
    verbose_feature_names_out = False,
)

So cot chu: 15


In [4]:
#xu ly tap train
preprocessor.fit(X_train)
feature_names = list(preprocessor.get_feature_names_out())
X_train_p = pd.DataFrame(preprocessor.transform(X_train), columns = feature_names, index=X_train.index)
X_test_p = pd.DataFrame(preprocessor.transform(X_test), columns = feature_names, index = X_test.index)
print("train:", X_train_p.shape, "|Test:", X_test_p.shape)
print("So gia tri thieu:", X_train_p.isna().sum(), X_test_p.isna().sum().sum())


train: (5634, 45) |Test: (1409, 45)
So gia tri thieu: tenure                                     0
MonthlyCharges                             0
TotalCharges                               0
gender_Female                              0
gender_Male                                0
Partner_No                                 0
Partner_Yes                                0
Dependents_No                              0
Dependents_Yes                             0
PhoneService_No                            0
PhoneService_Yes                           0
MultipleLines_No                           0
MultipleLines_No phone service             0
MultipleLines_Yes                          0
InternetService_DSL                        0
InternetService_Fiber optic                0
InternetService_No                         0
OnlineSecurity_No                          0
OnlineSecurity_No internet service         0
OnlineSecurity_Yes                         0
OnlineBackup_No                            0
O

In [5]:
print("Train - trung binh:", X_train_p[num_cols].mean().round(3).to_dict())
print("Train - do lech chuan:", X_train_p[num_cols].std(ddof=0).round(3).to_dict())
print("Test -trung binh:", X_test_p[num_cols].mean().round(3).to_dict())

print(feature_names)

Train - trung binh: {'tenure': -0.0, 'MonthlyCharges': -0.0, 'TotalCharges': -0.0}
Train - do lech chuan: {'tenure': 1.0, 'MonthlyCharges': 1.0, 'TotalCharges': 1.0}
Test -trung binh: {'tenure': -0.023, 'MonthlyCharges': -0.028, 'TotalCharges': -0.043}
['tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Female', 'gender_Male', 'Partner_No', 'Partner_Yes', 'Dependents_No', 'Dependents_Yes', 'PhoneService_No', 'PhoneService_Yes', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No', 'Stream

In [7]:
from sklearn.linear_model import LogisticRegression #mo dinh tuyen tinh don gian
from sklearn.ensemble import RandomForestClassifier # nhieu cay quyet dinh saong song
from xgboost import XGBClassifier # mo hinh nhieu cay quyet dinh noi tiep nhau

models = {
    "LogisticRegression": LogisticRegression(
        max_iter=2000, #so vong lap toi da
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, #so cay trong rung
        max_depth=10, #do sau toi da cua moi cay
        min_samples_leaf=5, #moi la phai co it nhat 5 khach
        random_state=42,
        n_jobs=-1, #
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, # so cac cay them noi tiep nhau
        max_depth=4, 
        learning_rate=0.05, #moi cay chi sua mot phan nho cua cay trong giup hoc cham maf chawc
        subsample=0.8, #moi cay chi nhin 80% so dong
        colsample_bytree=0.8, #moi cay chi nhin 80% so cot
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
    ),
}


In [8]:
for name, model in models.items():
    model.fit(X_train_p, y_train)
    print(name,"da train xong")

LogisticRegression da train xong
RandomForest da train xong
XGBoost da train xong


In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
results = []
for name, model in models.items():
    proba = model.predict_proba(X_test_p)[:, 1]
    pred = (proba >= 0.5).astype(int)
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
    })
results_df = pd.DataFrame(results).set_index("model").round(4)
results_df


,accuracy,precision,recall,f1,roc_auc
model,,,,,
LogisticRegression,0.8055,0.6572,0.5588,0.6040,0.8420
RandomForest,0.8062,0.6760,0.5187,0.5870,0.8420
XGBoost,0.7977,0.6498,0.5160,0.5753,0.8402


In [ ]:
from sklearn.metrics import confusion_matrix

pred_lr = models["LogisticRegression"].predict(X_test_p)
print(confusion_matrix(y_test, pred_lr))
# ma tran nham lan
#dang [[doan dung "o lai", doan dung "roi di"]],
#       [bo sot khach roi di, doan dung "roi di"]
print("Accurcary neu luon doan 'o lai':", round(1-y_test.mean(), 4))

[[926 109]
 [165 209]]
Accurcary neu luon doan 'o lai': 0.7346
